In [4]:
# bbtransformer_analyzer.py 

import os
from pathlib import Path
from typing import List, Optional, Dict, Any, Union
from bbtransformer import run_analysis


NEURO_XCONFIG = {
    'feature_dim': 414,
    'num_classes': 1,
    'embed_dim': 512,
    'num_heads': 8,
    'num_layers': 6,
    'n_kv_heads': 4,
    'embed_dim_age': 32,
    'embed_dim_ext': 16,
    'patch_size': 3,
    'patch_embed_ratio': 0.5,
    'temp_attn_hidden': 128,
    'dropout_input': 0.27,
    'dropout_patch': 0.27,
    'dropout_attn': 0.146,
    'dropout_ffn': 0.275,
    'dropout_classifier': 0.029,
    'dropout_temporal': 0.167,
    'stochastic_depth_rate': 0.1,
    'return_attn_weights': False,
}

TRAIN_PARAMS = {
    'epochs': 5000,
    'lr': 2.3157e-05,
    'weight_decay': 1.14e-06,
    'patience': 90
}


class BBTransformerAnalyzer:
    def __init__(
        self,
        base_dir: str,
        weights_dir: str = "weights",
        results_dir: str = "results",
        initial_weights: Optional[str] = None,
        min_composite: float = 0.60,
        max_trials_per_disorder: int = 50
    ):
        self.base_dir = Path(base_dir)
        self.weights_dir = Path(weights_dir)
        self.results_dir = Path(results_dir)
        self.weights_dir.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        self.current_weights = initial_weights
        self.min_composite = min_composite
        self.max_trials = max_trials_per_disorder
        self.valid_models = []

    def get_chrt_paths(self, disorder: str):
        """Resolve CHRT-style paths."""
        return (
            self.base_dir / f"fmri_{disorder}.npz",
            self.base_dir / f"pheno_{disorder}.csv"
        )

    def resolve_paths(self, task_spec: Union[str, Dict[str, str]]):
        """
        Resolve data paths from either:
          - str: disorder name → use CHRT convention
          - dict: {'data_path': ..., 'pheno_path': ...}
        """
        if isinstance(task_spec, str):
            # Assume CHRT-style disorder name
            return self.get_chrt_paths(task_spec)
        elif isinstance(task_spec, dict):
            # Explicit paths
            if 'data_path' not in task_spec or 'pheno_path' not in task_spec:
                raise ValueError("Dict must contain 'data_path' and 'pheno_path'")
            return Path(task_spec['data_path']), Path(task_spec['pheno_path'])
        else:
            raise TypeError("task_spec must be str or dict")

    def is_valid(self, metrics: Dict[str, float]) -> bool:
        return all(
            metrics.get(metric, 0) >= self.min_composite
            for metric in ['f1', 'roc_auc', 'accuracy', 'precision', 'recall']
        )

    def run_ordered_pipeline(self, tasks: List[Union[str, Dict[str, str]]]) -> Dict[str, Any]:
        """
        Run pipeline over mixed task specs.
        Each task can be:
          - str: e.g., 'NervousSystem_Other_Neuro'
          - dict: e.g., {'data_path': '/abide/...', 'pheno_path': '/abide/...', 'name': 'ASD'}
        """
        results_summary = {}

        for i, task in enumerate(tasks, 1):
            # Extract display name
            if isinstance(task, str):
                disorder_name = task
            else:
                disorder_name = task.get('name', 'unnamed_task')

            print(f"\n{'='*70}")
            print(f"PHASE {i}/{len(tasks)}: {disorder_name}")
            print(f"{'='*70}")

            data_path, pheno_path = self.resolve_paths(task)
            
            if not data_path.exists():
                print(f"  ❌ Data not found: {data_path}")
                continue
            if not pheno_path.exists():
                print(f"  ❌ Phenotype not found: {pheno_path}")
                continue

            use_pretrained = self.current_weights is not None
            best_result = None

            for trial in range(self.max_trials):
                print(f"  Trial {trial+1}/{self.max_trials}...")

                try:
                    result = run_analysis(
                        model_config=NEURO_XCONFIG,
                        training_config=TRAIN_PARAMS,
                        target_column=disorder_name,
                        data_path=str(data_path),
                        pheno_path=str(pheno_path),
                        use_pretrained=use_pretrained,
                        pretrained_weight_file=self.current_weights,
                        compute_importance=False,
                        random_seed=42 + trial,
                        weights_dir=str(self.weights_dir)
                    )
                    
                    if self.is_valid(result['metrics']):
                        best_result = result
                        print(f"  ✅ VALID MODEL FOUND (Composite: {result['metrics']['f1']:.4f})")
                        break
                    else:
                        print(f"  ❌ Trial {trial+1} failed validity check")
                        
                except Exception as e:
                    print(f"  ❌ Trial {trial+1} crashed: {str(e)}")
                    continue

            results_summary[disorder_name] = {
                'valid': best_result is not None,
                'metrics': best_result['metrics'] if best_result else None,
                'weights_used': self.current_weights,
                'weights_saved': None
            }

            if best_result is not None:
                weight_file = f"weights_{disorder_name}.pth"
                self.current_weights = str(self.weights_dir / weight_file)
                results_summary[disorder_name]['weights_saved'] = self.current_weights
                self.valid_models.append(disorder_name)
                print(f"  🔁 Propagating weights to next disorder")
            else:
                if self.valid_models:
                    print(f"  ⚠️ Keeping weights from last valid model: {self.valid_models[-1]}")
                else:
                    print(f"  🧼 No prior valid model—next disorder will train from scratch")
                    self.current_weights = None

        return results_summary

In [5]:
TASKS = [
    # Vascular & inflammatory (network disruption)
    'NervousSystem_Cerebrovascular',
    'NervousSystem_Inflammatory_Infectious',
    'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
    
    # Epilepsy & paroxysmal (temporal instability)
    'NervousSystem_Epilepsy_Status_Epilepticus',
    
    # Psychiatric (distributed, heterogeneous)
    'ICD_F32_Depressive_Episode',
    'ICD_F31_Bipolar',
    'Psychopathology_Mood_Affective',
    'Psychopathology_Schizophrenia_Spectrum',
    'Psychopathology_Substance_Use',
    'Psychopathology_Organic_Mental_Disorder',
    
    # Broad neurological control
    'NervousSystem_Sleep_Disorders',
    'NervousSystem_Other_Neuro',


   # Neurodegenerative (focal, high signal)
    'NervousSystem_Parkinsons_Other_Movement',
    'NervousSystem_Dementia_Developmental',
    'Psychopathology_Dementia',
    
    # External datasets (explicit paths)
    {
        'name': 'ASD',
        'data_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz',
        'pheno_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv'
    },
    {
        'name': 'ADHD',
        'data_path': '/mnt/movement/users/jaizor/xtra/data/fmri/adhd/fmri_ADHD_all_runs.npz',
        'pheno_path': '/mnt/movement/users/jaizor/xtra/data/fmri/adhd/pheno_ADHD_all_runs.csv'
    }
]



In [6]:

analyzer = BBTransformerAnalyzer(
    base_dir='/mnt/movement/users/jaizor/xtra/data/fmri/chrt',
    weights_dir='/mnt/movement/users/jaizor/xtra/ΞΞ/__/weights', 
    initial_weights='weights_ADHD.pth',  
    min_composite=0.7,
    max_trials_per_disorder=5
)

results = analyzer.run_ordered_pipeline(TASKS)


PHASE 1/17: NervousSystem_Cerebrovascular
  Trial 1/5...
STEP 1: Loading Data for Target = 'NervousSystem_Cerebrovascular'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Cerebrovascular.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Cerebrovascular.csv
Loaded phenotype: (592, 56)
Loaded fMRI: (592, 150, 414)
  Subjects: 592
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 592 subjects (296 cases, 296 controls, 50.0% prevalence)
Splits → Train: 414, Val: 89, Test: 89

Dataset Meta
  target: NervousSystem_Cerebrovascular
  n_total: 592
  n_positive: 296
  prevalence: 0.5
  feature_dim: 414
  n_train: 414
  n_val: 89
  n_test: 89

STEP 3: Initializing BBTransformer
Model created on cuda with 25,878,528 parameters

STEP 3.5: Loading Pretrained Weights
  From: /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/weights_ADHD.pth
Attempting SAFE load (on CPU 

Early stopping at epoch 221 (F1: 0.5833)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Cerebrovascular.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.7640
  Precision: 0.7556
  Recall:    0.7727
  F1 Score:  0.7640
  ROC-AUC:   0.8293

Confusion Matrix:
[[34 11]
 [10 34]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Cerebrovascular_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Cerebrovascular
  ✅ VALID MODEL FOUND (Composite: 0.7640)
  🔁 Propagating weights to next disorder

PHASE 2/17: NervousSystem_Inflammatory_Infectious
  Trial 1/5...
STEP 1: Loading Data for Target = 'NervousSystem_Inflammatory_Infectious'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Inflammatory_Infectious.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Inflammatory_Infectious.csv
Loaded phenotype: (92, 56)
Loaded fMRI: (92, 150, 414)
  Subjects: 92
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort:

Early stopping at epoch 91 (F1: 0.9231)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Inflammatory_Infectious.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.9286
  Precision: 1.0000
  Recall:    0.8571
  F1 Score:  0.9231
  ROC-AUC:   1.0000

Confusion Matrix:
[[7 0]
 [1 6]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Inflammatory_Infectious_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Inflammatory_Infectious
  ✅ VALID MODEL FOUND (Composite: 0.9231)
  🔁 Propagating weights to next disorder

PHASE 3/17: NervousSystem_Multiple_Sclerosis_Other_Demyelinating
  Trial 1/5...
STEP 1: Loading Data for Target = 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.csv
Loaded phenotype: (164, 56)
Loaded fMRI: (164, 150, 414)
  Subjects: 164
  Timepoints: 150
  Brain regions: 414
  ROI label

...


Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000
  ROC-AUC:   1.0000

Confusion Matrix:
[[8 0]
 [0 7]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/Psychopathology_Dementia_results.json

TRAINING & EVALUATION COMPLETE
Target: Psychopathology_Dementia
  ✅ VALID MODEL FOUND (Composite: 1.0000)
  🔁 Propagating weights to next disorder

PHASE 16/17: ASD
  Trial 1/5...
STEP 1: Loading Data for Target = 'ASD'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv
Loaded phenotype: (585, 4)
Loaded fMRI: (585, 150, 414)
  Subjects: 585
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 585 subjects (271 cases, 314 controls, 46.3% prevalence)
Splits → Train: 409, Val: 88, Test: 88

Dataset Meta
  target: ASD
  n_total: 585
  n_p

Early stopping at epoch 152 (F1: 0.7887)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_ASD.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8864
  Precision: 0.8780
  Recall:    0.8780
  F1 Score:  0.8780
  ROC-AUC:   0.9520

Confusion Matrix:
[[42  5]
 [ 5 36]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/ASD_results.json

TRAINING & EVALUATION COMPLETE
Target: ASD
  ✅ VALID MODEL FOUND (Composite: 0.8780)
  🔁 Propagating weights to next disorder

PHASE 17/17: ADHD
  Trial 1/5...
STEP 1: Loading Data for Target = 'ADHD'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/adhd/fmri_ADHD_all_runs.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/adhd/pheno_ADHD_all_runs.csv
Loaded phenotype: (242, 6)
Loaded fMRI: (242, 150, 414)
  Subjects: 242
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 242 subjects (103 cases, 139 controls, 42.6% prevalence)
Splits → Train: 169, Val: 36, Test: 37

Dataset Meta
  target: ADHD
  n_total: 242
  n_positive: 103
  pr

Early stopping at epoch 163 (F1: 0.7647)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_ADHD.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.9730
  Precision: 1.0000
  Recall:    0.9375
  F1 Score:  0.9677
  ROC-AUC:   0.9881

Confusion Matrix:
[[21  0]
 [ 1 15]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/ADHD_results.json

TRAINING & EVALUATION COMPLETE
Target: ADHD
  ✅ VALID MODEL FOUND (Composite: 0.9677)
  🔁 Propagating weights to next disorder
